# NYC 311 Service Requests: Interactive Data Quality & Pipeline Exploration (Part A)
**Student:** Dilawar khan (Roll No: TRN-2026-000961)
**Date:** 17 Nov 2026
**Dataset:** NYC 311 Service Requests (2010 - 2019, 20,000 Stratified Rows)

This notebook provides an interactive walkthrough of Part A (Checkpoints 0 - 3):
1. Raw multi-year data inspection
2. Modular cleaning, negative timestamp separation into `rejected_rows.csv`, and Quality Gate
3. Vectorized feature engineering (NumPy) and statistical summaries (Pandas)
4. Relational SQLite demographic JOIN query

In [ ]:
import os
import sys
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sqlite3

from src.extract import extract_raw_data
from src.clean import clean_dataset
from src.validate import run_quality_gate
from src.features import add_engineered_features
from src.database import run_sqlite_pipeline

## 1. Inspect Raw Data
Load untouched raw API extract from `../data/raw/raw_311_data.csv`.

In [ ]:
raw_df = pd.read_csv('../data/raw/raw_311_data.csv', low_memory=False)
print(f'Raw dataset shape: {raw_df.shape[0]:,} rows x {raw_df.shape[1]} columns')
print('Years present:', sorted(pd.to_datetime(raw_df['created_date']).dt.year.unique()))
raw_df.head(3)

## 2. Execute Cleaning and Quality Gate
Run cleaning transformations, separate rejected invalid records, and verify Quality Gate.

In [ ]:
cleaned_df, rejected_df = clean_dataset(raw_df, rejected_output_dir='../outputs/reports')
enriched_df = add_engineered_features(cleaned_df)
metrics = run_quality_gate(enriched_df, min_pass_rate=0.95)
print(f"Cleaned records: {len(enriched_df):,} | Rejected invalid: {len(rejected_df):,}")
print(f"Quality Gate Pass Rate: {metrics['validation_pass_rate']*100:.2f}%")

## 3. Query Embedded SQLite Database with Demographic JOIN

In [ ]:
conn = sqlite3.connect('../data/nyc_311.db')
query = '''
SELECT 
    b.borough AS borough_name,
    b.population AS census_population,
    COUNT(t.unique_key) AS total_requests,
    ROUND((COUNT(t.unique_key) * 10000.0) / b.population, 2) AS requests_per_10k_residents,
    ROUND(AVG(t.resolution_hours), 2) AS avg_resolution_hours,
    ROUND(SUM(CASE WHEN t.is_zero_duration = 1 THEN 1.0 ELSE 0.0 END) * 100.0 / COUNT(t.unique_key), 2) AS pct_zero_duration
FROM borough_lookup b
INNER JOIN tickets t ON b.borough = t.borough
GROUP BY b.borough, b.population
ORDER BY requests_per_10k_residents DESC;
'''
df_sql = pd.read_sql_query(query, conn)
conn.close()
df_sql